In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
!pip install --quiet sqlalchemy scikit-learn sentence-transformers

import sqlite3, json, time, os
from typing import List, Dict, Any, Callable
from datetime import datetime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 83.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 69.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 27.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 73.0 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installe

In [2]:
# --- Simple LLM wrapper (replace with real API) ---
def llm_call(prompt: str, temperature: float=0.0) -> str:
    return "LLM response to: " + prompt[:200]

def web_search_tool(query: str) -> str:
    return f"Search results summary for '{query}' (placeholder)"

class StatefulAgent:
    def __init__(self, db_path="agent_memory.sqlite"):
        self.db_path = db_path
        self._init_db()
        self.tools = {"search": web_search_tool}

    def _init_db(self):
        self.conn = sqlite3.connect(self.db_path, check_same_thread=False)
        cur = self.conn.cursor()
        cur.execute("""
        CREATE TABLE IF NOT EXISTS messages (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            role TEXT,
            content TEXT,
            ts TEXT
        )""")
        cur.execute("""
        CREATE TABLE IF NOT EXISTS facts (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            key TEXT UNIQUE,
            value TEXT,
            ts TEXT
        )""")
        self.conn.commit()

    def add_message(self, role: str, content: str):
        cur = self.conn.cursor()
        cur.execute("INSERT INTO messages (role, content, ts) VALUES (?, ?, ?)",
                    (role, content, datetime.utcnow().isoformat()))
        self.conn.commit()

    def get_recent_messages(self, limit=10) -> List[Dict[str,str]]:
        cur = self.conn.cursor()
        cur.execute("SELECT role, content, ts FROM messages ORDER BY id DESC LIMIT ?", (limit,))
        rows = cur.fetchall()
        return [{"role": r[0], "content": r[1], "ts": r[2]} for r in rows][::-1]

    def set_fact(self, key: str, value: str):
        cur = self.conn.cursor()
        cur.execute("INSERT OR REPLACE INTO facts (key, value, ts) VALUES (?, ?, ?)",
                    (key, value, datetime.utcnow().isoformat()))
        self.conn.commit()

    def get_fact(self, key: str):
        cur = self.conn.cursor()
        cur.execute("SELECT value FROM facts WHERE key = ?", (key,))
        row = cur.fetchone()
        return row[0] if row else None

    def call_tool(self, tool_name: str, arg: str) -> str:
        if tool_name not in self.tools:
            return f"Unknown tool: {tool_name}"
        return self.tools[tool_name](arg)

    def handle_user(self, user_text: str) -> str:
        self.add_message("user", user_text)

        if user_text.lower().startswith("search:"):
            query = user_text.split(":",1)[1].strip()
            tool_out = self.call_tool("search", query)
            self.add_message("assistant", f"[tool:search] {tool_out}")
            return tool_out

        recent = self.get_recent_messages(limit=8)
        facts = self._fetch_all_facts()
        prompt_parts = ["You are a helpful assistant. Keep answers concise."]
        if facts:
            prompt_parts.append("Facts:")
            for k,v in facts.items():
                prompt_parts.append(f"- {k}: {v}")
        prompt_parts.append("Conversation:")
        for m in recent:
            prompt_parts.append(f"{m['role']}: {m['content']}")

        prompt_parts.append("Assistant:") 
        prompt = "\n".join(prompt_parts)
        resp = llm_call(prompt)
        self.add_message("assistant", resp)
        return resp

    def _fetch_all_facts(self):
        cur = self.conn.cursor()
        cur.execute("SELECT key, value FROM facts")
        return {r[0]: r[1] for r in cur.fetchall()}

In [3]:
agent = StatefulAgent()
print(agent.handle_user("Hello, I'm Alex."))
agent.set_fact("name", "Alex")
print(agent.handle_user("Remember my favorite language. It's Python."))
print(agent.handle_user("Search: stateful agents tutorial"))
print("Recent messages:", agent.get_recent_messages())
print("Stored fact name:", agent.get_fact("name"))

LLM response to: You are a helpful assistant. Keep answers concise.
Conversation:
user: Hello, I'm Alex.
Assistant:
LLM response to: You are a helpful assistant. Keep answers concise.
Facts:
- name: Alex
Conversation:
user: Hello, I'm Alex.
assistant: LLM response to: You are a helpful assistant. Keep answers concise.
Conversation:
Search results summary for 'stateful agents tutorial' (placeholder)
Recent messages: [{'role': 'user', 'content': "Hello, I'm Alex.", 'ts': '2025-11-12T11:56:19.671911'}, {'role': 'assistant', 'content': "LLM response to: You are a helpful assistant. Keep answers concise.\nConversation:\nuser: Hello, I'm Alex.\nAssistant:", 'ts': '2025-11-12T11:56:19.678971'}, {'role': 'user', 'content': "Remember my favorite language. It's Python.", 'ts': '2025-11-12T11:56:19.693345'}, {'role': 'assistant', 'content': "LLM response to: You are a helpful assistant. Keep answers concise.\nFacts:\n- name: Alex\nConversation:\nuser: Hello, I'm Alex.\nassistant: LLM response to:

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
import numpy as np

docs = [
    "Stateful agents store memory and can call external tools like search and calculators.",
    "Context engineering uses prompt templates, few-shot examples, and retrieval.",
    "When using large documents, chunk them and store embeddings for retrieval.",
    "Use system messages to constrain model behavior: be concise and avoid hallucination."
]
doc_ids = list(range(len(docs)))
vectorizer = TfidfVectorizer().fit(docs)
doc_vectors = vectorizer.transform(docs)

def retrieve(query: str, top_k=2):
    qv = vectorizer.transform([query])
    sims = linear_kernel(qv, doc_vectors).flatten()
    top_idx = sims.argsort()[::-1][:top_k]
    return [(i, docs[i], float(sims[i])) for i in top_idx if sims[i] > 0]

SYSTEM_MSG = "You are a concise helpful assistant. Use only the provided context to answer."
FEW_SHOT = [
    {"q":"What is a stateful agent?", "a":"An agent that keeps memory over time and can recall prior interactions."},
    {"q":"How do we reduce hallucinations?", "a":"By providing only verified retrieved context and instructing the model to say 'I don't know' when unsure."}
]

def assemble_prompt(user_q: str, retrieved_snippets: List[str]):
    parts = [SYSTEM_MSG, "\n\nContext:"]
    for s in retrieved_snippets:
        parts.append(s)
    parts.append("\n\nExamples:")
    for ex in FEW_SHOT:
        parts.append(f"Q: {ex['q']}\nA: {ex['a']}")
    parts.append("\n\nUser query:\n" + user_q)
    parts.append("\nAnswer concisely using only the context above.")
    return "\n".join(parts)

query = "How do agents remember previous interactions?"
hits = retrieve(query, top_k=3)
snippets = [h[1] for h in hits]
prompt = assemble_prompt(query, snippets)
print("PROMPT:\n", prompt[:1000], "...\n")
print("LLM OUTPUT:", llm_call(prompt))

PROMPT:
 You are a concise helpful assistant. Use only the provided context to answer.


Context:
Stateful agents store memory and can call external tools like search and calculators.


Examples:
Q: What is a stateful agent?
A: An agent that keeps memory over time and can recall prior interactions.
Q: How do we reduce hallucinations?
A: By providing only verified retrieved context and instructing the model to say 'I don't know' when unsure.


User query:
How do agents remember previous interactions?

Answer concisely using only the context above. ...

LLM OUTPUT: LLM response to: You are a concise helpful assistant. Use only the provided context to answer.


Context:
Stateful agents store memory and can call external tools like search and calculators.


Examples:
Q: What is a 


In [6]:
def build_prompt_with_budget(user_q: str, retrieved: List[str], budget_chars=2000):
    sys = SYSTEM_MSG + "\n\n"
    examples = "\n".join([f"Q: {e['q']}\nA: {e['a']}" for e in FEW_SHOT]) + "\n\n"
    base = sys + "Context:\n"
    out = base
    for s in retrieved:
        if len(out) + len(s) + len(examples) + len(user_q) > budget_chars:
            break
        out += s + "\n---\n"
    out += examples + "User query:\n" + user_q + "\nAnswer:"
    return out

In [7]:
def test_answer(query):
    hits = retrieve(query, top_k=2)
    snippets = [h[1] for h in hits]
    prompt = assemble_prompt(query, snippets)
    resp = llm_call(prompt)
    print("Q:", query)
    print("A:", resp)

test_answer("What techniques reduce hallucination?")
test_answer("Who invented the jet engine?")  

Q: What techniques reduce hallucination?
A: LLM response to: You are a concise helpful assistant. Use only the provided context to answer.


Context:
Use system messages to constrain model behavior: be concise and avoid hallucination.


Examples:
Q: What is a s
Q: Who invented the jet engine?
A: LLM response to: You are a concise helpful assistant. Use only the provided context to answer.


Context:


Examples:
Q: What is a stateful agent?
A: An agent that keeps memory over time and can recall prior interacti


In [9]:
!pip install --quiet openai sentence-transformers tiktoken sqlalchemy
import os, sqlite3, json, time
from datetime import datetime
from typing import List, Dict, Any

In [10]:
import openai

def llm_call(prompt: str, temperature: float = 0.4, max_tokens: int = 400) -> str:
    """Adaptive LLM call with fallback."""
    api_key = os.getenv("OPENAI_API_KEY")
    if api_key:
        openai.api_key = api_key
        try:
            resp = openai.ChatCompletion.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
                max_tokens=max_tokens,
            )
            return resp.choices[0].message["content"].strip()
        except Exception as e:
            return f"[LLM error: {e}]"
    else:
        return "Offline LLM: " + prompt[:200]

In [12]:
class StatefulAgent:
    def __init__(self, db_path="agent_memory.sqlite"):
        self.db_path = db_path
        self._init_db()
        self.tools = {"search": self.web_search_tool, "calc": self.calc_tool}
        self.summary_threshold = 10  # summarize after 10 messages

    def _init_db(self):
        self.conn = sqlite3.connect(self.db_path, check_same_thread=False)
        cur = self.conn.cursor()
        cur.execute("""CREATE TABLE IF NOT EXISTS memory(id INTEGER PRIMARY KEY, role TEXT, content TEXT, ts TEXT)""")
        cur.execute("""CREATE TABLE IF NOT EXISTS facts(key TEXT UNIQUE, value TEXT, ts TEXT)""")
        self.conn.commit()

    def add_message(self, role, content):
        cur = self.conn.cursor()
        cur.execute("INSERT INTO memory(role,content,ts) VALUES(?,?,?)",
                    (role, content, datetime.utcnow().isoformat()))
        self.conn.commit()

    def get_recent(self, limit=10):
        cur = self.conn.cursor()
        cur.execute("SELECT role,content FROM memory ORDER BY id DESC LIMIT ?", (limit,))
        return cur.fetchall()[::-1]

    def set_fact(self, key, value):
        cur = self.conn.cursor()
        cur.execute("INSERT OR REPLACE INTO facts(key,value,ts) VALUES(?,?,?)",
                    (key, value, datetime.utcnow().isoformat()))
        self.conn.commit()

    def get_facts(self):
        cur = self.conn.cursor()
        cur.execute("SELECT key,value FROM facts")
        return dict(cur.fetchall())

    def web_search_tool(self, query):
        return f"[search results for '{query}' — placeholder summary]"
    def calc_tool(self, expr):
        try:
            return str(eval(expr))
        except Exception as e:
            return f"[calc error: {e}]"

    def maybe_summarize(self):
        cur = self.conn.cursor()
        cur.execute("SELECT COUNT(*) FROM memory")
        count = cur.fetchone()[0]
        if count > self.summary_threshold:
            msgs = self.get_recent(limit=self.summary_threshold)
            convo = "\n".join([f"{r}: {c}" for r,c in msgs])
            summary = llm_call(f"Summarize this conversation briefly:\n{convo}")
            self.add_message("system", f"[summary] {summary}")
            cur.execute("DELETE FROM memory WHERE id NOT IN (SELECT id FROM memory ORDER BY id DESC LIMIT 5)")
            self.conn.commit()

    def handle_user(self, text):
        self.add_message("user", text)
        self.maybe_summarize()

        if text.lower().startswith("search:"):
            q = text.split(":",1)[1].strip()
            out = self.web_search_tool(q)
            self.add_message("assistant", out)
            return out
        if text.lower().startswith("calc:"):
            expr = text.split(":",1)[1].strip()
            out = self.calc_tool(expr)
            self.add_message("assistant", out)
            return out

        facts = self.get_facts()
        convo = "\n".join([f"{r}: {c}" for r,c in self.get_recent()])
        prompt = f"You are a concise helpful AI.\nFacts: {facts}\nConversation:\n{convo}\nUser: {text}\nAssistant:"
        reply = llm_call(prompt)
        self.add_message("assistant", reply)
        return reply

In [13]:
agent = StatefulAgent()

print(agent.handle_user("Hi, I’m Hamsa!"))
agent.set_fact("user_name", "Hamsa")
print(agent.handle_user("search: stateful agents in AI"))
print(agent.handle_user("calc: (3+7)*2"))
print(agent.handle_user("Can you remember my name?"))

Offline LLM: You are a concise helpful AI.
Facts: {'name': 'Alex'}
Conversation:
user: Hi, I’m Hamsa!
User: Hi, I’m Hamsa!
Assistant:
[search results for 'stateful agents in AI' — placeholder summary]
20
Offline LLM: You are a concise helpful AI.
Facts: {'name': 'Alex', 'user_name': 'Hamsa'}
Conversation:
user: Hi, I’m Hamsa!
assistant: Offline LLM: You are a concise helpful AI.
Facts: {'name': 'Alex'}
Conversatio


In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

docs = [
    "Stateful agents maintain long-term memory using databases or embeddings.",
    "Context engineering involves prompt design, retrieval, and few-shot learning.",
    "Few-shot examples demonstrate correct response patterns to guide the model.",
    "Summarization helps reduce token usage and maintain relevant context."
]

vectorizer = TfidfVectorizer().fit(docs)
doc_vecs = vectorizer.transform(docs)

def retrieve(query, top_k=2):
    qv = vectorizer.transform([query])
    sims = linear_kernel(qv, doc_vecs).flatten()
    idxs = sims.argsort()[::-1][:top_k]
    return [docs[i] for i in idxs]

def smart_prompt(query):
    context = "\n".join(retrieve(query))
    return f"You are helpful. Use only this context:\n{context}\n\nQ: {query}\nA:"

print(agent.handle_user("Explain few-shot learning"))

Offline LLM: You are a concise helpful AI.
Facts: {'name': 'Alex', 'user_name': 'Hamsa'}
Conversation:
user: Hi, I’m Hamsa!
assistant: Offline LLM: You are a concise helpful AI.
Facts: {'name': 'Alex'}
Conversatio


In [15]:
print("Facts:", agent.get_facts())
print("Recent messages:", agent.get_recent())
agent2 = StatefulAgent("agent_memory.sqlite")
print("Restored facts:", agent2.get_facts())

Facts: {'name': 'Alex', 'user_name': 'Hamsa'}
Recent messages: [('user', 'Hi, I’m Hamsa!'), ('assistant', "Offline LLM: You are a concise helpful AI.\nFacts: {'name': 'Alex'}\nConversation:\nuser: Hi, I’m Hamsa!\nUser: Hi, I’m Hamsa!\nAssistant:"), ('user', 'search: stateful agents in AI'), ('assistant', "[search results for 'stateful agents in AI' — placeholder summary]"), ('user', 'calc: (3+7)*2'), ('assistant', '20'), ('user', 'Can you remember my name?'), ('assistant', "Offline LLM: You are a concise helpful AI.\nFacts: {'name': 'Alex', 'user_name': 'Hamsa'}\nConversation:\nuser: Hi, I’m Hamsa!\nassistant: Offline LLM: You are a concise helpful AI.\nFacts: {'name': 'Alex'}\nConversatio"), ('user', 'Explain few-shot learning'), ('assistant', "Offline LLM: You are a concise helpful AI.\nFacts: {'name': 'Alex', 'user_name': 'Hamsa'}\nConversation:\nuser: Hi, I’m Hamsa!\nassistant: Offline LLM: You are a concise helpful AI.\nFacts: {'name': 'Alex'}\nConversatio")]
Restored facts: {'nam

In [16]:
!pip install --quiet sentence-transformers scikit-learn > /dev/null 2>&1 || true

import os, sqlite3, json, time
from datetime import datetime
from typing import List, Dict, Any
print("Environment ready. Python version:", os.sys.version.splitlines()[0])

Environment ready. Python version: 3.11.13 (main, Jun  4 2025, 08:57:29) [GCC 11.4.0]


In [18]:
def llm_call(prompt: str, temperature: float = 0.0, max_tokens: int = 400) -> str:
    """
    Simple offline LLM fallback: returns a concise synthetic reply based on the prompt.
    Designed for testing the agent's structure, memory, and tools without external APIs.
    """
    text = prompt.strip().replace("\n", " ")
    short = text[:500]
    reply = f"[offline-echo] I received a prompt of {len(text)} chars. Preview: {short}"
    return reply

In [20]:
class StatefulAgent:
    def __init__(self, db_path="agent_memory.sqlite"):
        self.db_path = db_path
        self._init_db()
        self.tools = {"search": self.web_search_tool, "calc": self.calc_tool}
        self.summary_threshold = 10  # summarize after X messages

    def _init_db(self):
        self.conn = sqlite3.connect(self.db_path, check_same_thread=False)
        cur = self.conn.cursor()
        cur.execute("""CREATE TABLE IF NOT EXISTS memory(
                        id INTEGER PRIMARY KEY AUTOINCREMENT,
                        role TEXT,
                        content TEXT,
                        ts TEXT)""")
        cur.execute("""CREATE TABLE IF NOT EXISTS facts(
                        key TEXT UNIQUE,
                        value TEXT,
                        ts TEXT)""")
        self.conn.commit()

    def add_message(self, role, content):
        cur = self.conn.cursor()
        cur.execute("INSERT INTO memory(role,content,ts) VALUES(?,?,?)",
                    (role, content, datetime.utcnow().isoformat()))
        self.conn.commit()

    def get_recent(self, limit=10):
        cur = self.conn.cursor()
        cur.execute("SELECT role,content FROM memory ORDER BY id DESC LIMIT ?", (limit,))
        rows = cur.fetchall()[::-1]
        return [(r,c) for r,c in rows]

    def set_fact(self, key, value):
        cur = self.conn.cursor()
        cur.execute("INSERT OR REPLACE INTO facts(key,value,ts) VALUES(?,?,?)",
                    (key, value, datetime.utcnow().isoformat()))
        self.conn.commit()

    def get_facts(self):
        cur = self.conn.cursor()
        cur.execute("SELECT key,value FROM facts")
        return dict(cur.fetchall())
        
    def web_search_tool(self, query):
        return f"[search-placeholder] Top results summary for: {query}"

    def calc_tool(self, expr):
        try:
            allowed_names = {}
            result = eval(expr, {"__builtins__":None}, allowed_names)
            return str(result)
        except Exception as e:
            return f"[calc error] {e}"

    def maybe_summarize(self):
        cur = self.conn.cursor()
        cur.execute("SELECT COUNT(*) FROM memory")
        count = cur.fetchone()[0]
        if count > self.summary_threshold:
            msgs = self.get_recent(limit=self.summary_threshold)
            convo = "\n".join([f"{r}: {c}" for r,c in msgs])
            summary = llm_call(f"Summarize this conversation briefly (one or two sentences):\n{convo}")
            # store summary as a system message and keep only last 5 raw messages
            self.add_message("system", f"[summary] {summary}")
            # delete old messages except last 5
            cur.execute("DELETE FROM memory WHERE id NOT IN (SELECT id FROM memory ORDER BY id DESC LIMIT 5)")
            self.conn.commit()

    def handle_user(self, text):
        self.add_message("user", text)
        self.maybe_summarize()

        lt = text.lower()
        if lt.startswith("search:"):
            q = text.split(":",1)[1].strip()
            out = self.web_search_tool(q)
            self.add_message("assistant", out)
            return out
        if lt.startswith("calc:"):
            expr = text.split(":",1)[1].strip()
            out = self.calc_tool(expr)
            self.add_message("assistant", out)
            return out

        facts = self.get_facts()
        recent = self.get_recent(limit=8)
        convo = "\n".join([f"{r}: {c}" for r,c in recent])
        prompt = f"You are a concise helpful assistant. Facts: {facts} Conversation: {convo} User: {text} Assistant:"
        reply = llm_call(prompt)
        self.add_message("assistant", reply)
        return reply

In [21]:
agent = StatefulAgent()
print(agent.handle_user("Hello, I'm Alex."))
agent.set_fact("name", "Alex")
print(agent.handle_user("search: stateful agents tutorial"))
print(agent.handle_user("calc: (3+7)*2"))
print(agent.handle_user("Can you remember my name?"))
print("\nRecent messages:", agent.get_recent())
print("Stored facts:", agent.get_facts())

[offline-echo] I received a prompt of 1212 chars. Preview: You are a concise helpful assistant. Facts: {'name': 'Alex', 'user_name': 'Hamsa'} Conversation: assistant: Offline LLM: You are a concise helpful AI. Facts: {'name': 'Alex', 'user_name': 'Hamsa'} Conversation: user: Hi, I’m Hamsa! assistant: Offline LLM: You are a concise helpful AI. Facts: {'name': 'Alex'} Conversatio user: Explain few-shot learning assistant: Offline LLM: You are a concise helpful AI. Facts: {'name': 'Alex', 'user_name': 'Hamsa'} Conversation: user: Hi, I’m Hamsa! assistant: 
[search-placeholder] Top results summary for: stateful agents tutorial
20
[offline-echo] I received a prompt of 865 chars. Preview: You are a concise helpful assistant. Facts: {'user_name': 'Hamsa', 'name': 'Alex'} Conversation: assistant: [search-placeholder] Top results summary for: stateful agents tutorial user: calc: (3+7)*2 assistant: 20 user: Can you remember my name? system: [summary] [offline-echo] I received a prompt of 1674 ch

In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

docs = [
    "Stateful agents store memory and can call external tools like search and calculators.",
    "Context engineering uses prompt templates, few-shot examples, and retrieval.",
    "When using large documents, chunk them and store embeddings for retrieval.",
    "Use system messages to constrain model behavior: be concise and avoid hallucination."
]

vectorizer = TfidfVectorizer().fit(docs)
doc_vecs = vectorizer.transform(docs)

def retrieve(query, top_k=2):
    qv = vectorizer.transform([query])
    sims = linear_kernel(qv, doc_vecs).flatten()
    idxs = sims.argsort()[::-1][:top_k]
    return [docs[i] for i in idxs]

def assemble_prompt(user_q):
    context = "\n".join(retrieve(user_q, top_k=2))
    prompt = f"You are a helpful assistant. Use only this context:\n{context}\n\nQ: {user_q}\nA:"
    return prompt

q = "How do agents remember previous interactions?"
print("PROMPT PREVIEW:\n", assemble_prompt(q))
print("\nLLM REPLY:\n", llm_call(assemble_prompt(q)))

PROMPT PREVIEW:
 You are a helpful assistant. Use only this context:
Stateful agents store memory and can call external tools like search and calculators.
Use system messages to constrain model behavior: be concise and avoid hallucination.

Q: How do agents remember previous interactions?
A:

LLM REPLY:
 [offline-echo] I received a prompt of 275 chars. Preview: You are a helpful assistant. Use only this context: Stateful agents store memory and can call external tools like search and calculators. Use system messages to constrain model behavior: be concise and avoid hallucination.  Q: How do agents remember previous interactions? A:


In [23]:
print("Facts before:", agent.get_facts())
print("Recent before:", agent.get_recent())
agent2 = StatefulAgent("agent_memory.sqlite")
print("Restored facts:", agent2.get_facts())
print("Restored recent:", agent2.get_recent())

Facts before: {'user_name': 'Hamsa', 'name': 'Alex'}
Recent before: [('assistant', '[search-placeholder] Top results summary for: stateful agents tutorial'), ('user', 'calc: (3+7)*2'), ('assistant', '20'), ('user', 'Can you remember my name?'), ('system', "[summary] [offline-echo] I received a prompt of 1674 chars. Preview: Summarize this conversation briefly (one or two sentences): user: Explain few-shot learning assistant: Offline LLM: You are a concise helpful AI. Facts: {'name': 'Alex', 'user_name': 'Hamsa'} Conversation: user: Hi, I’m Hamsa! assistant: Offline LLM: You are a concise helpful AI. Facts: {'name': 'Alex'} Conversatio user: Hello, I'm Alex. system: [summary] [offline-echo] I received a prompt of 889 chars. Preview: Summarize this conversation briefly (one or two sentences): assistant: Offline LLM: "), ('assistant', "[offline-echo] I received a prompt of 865 chars. Preview: You are a concise helpful assistant. Facts: {'user_name': 'Hamsa', 'name': 'Alex'} Conversation: 

In [24]:
!pip install --quiet transformers sentence-transformers scikit-learn accelerate

import os, sqlite3, json, time
from datetime import datetime
from typing import List, Dict, Any
print("Environment ready.")

Environment ready.


In [25]:
from transformers import pipeline

generator = pipeline("text-generation", model="microsoft/phi-2", device_map="auto")

def llm_call(prompt: str, temperature: float = 0.7, max_new_tokens: int = 200) -> str:
    try:
        result = generator(prompt, max_new_tokens=max_new_tokens, temperature=temperature)
        return result[0]['generated_text'].split(prompt)[-1].strip()
    except Exception as e:
        return f"[HF model error: {e}]"

2025-11-12 12:33:32.420854: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762950812.674290      48 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762950812.744883      48 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

Device set to use cpu


In [26]:
class StatefulAgent:
    def __init__(self, db_path="agent_memory.sqlite"):
        self.db_path = db_path
        self._init_db()
        self.tools = {"search": self.web_search_tool, "calc": self.calc_tool}
        self.summary_threshold = 10

    def _init_db(self):
        self.conn = sqlite3.connect(self.db_path, check_same_thread=False)
        cur = self.conn.cursor()
        cur.execute("CREATE TABLE IF NOT EXISTS memory(id INTEGER PRIMARY KEY, role TEXT, content TEXT, ts TEXT)")
        cur.execute("CREATE TABLE IF NOT EXISTS facts(key TEXT UNIQUE, value TEXT, ts TEXT)")
        self.conn.commit()

    def add_message(self, role, content):
        cur = self.conn.cursor()
        cur.execute("INSERT INTO memory(role,content,ts) VALUES(?,?,?)",
                    (role, content, datetime.utcnow().isoformat()))
        self.conn.commit()

    def get_recent(self, limit=10):
        cur = self.conn.cursor()
        cur.execute("SELECT role,content FROM memory ORDER BY id DESC LIMIT ?", (limit,))
        return cur.fetchall()[::-1]

    def set_fact(self, key, value):
        cur = self.conn.cursor()
        cur.execute("INSERT OR REPLACE INTO facts(key,value,ts) VALUES(?,?,?)",
                    (key, value, datetime.utcnow().isoformat()))
        self.conn.commit()

    def get_facts(self):
        cur = self.conn.cursor()
        cur.execute("SELECT key,value FROM facts")
        return dict(cur.fetchall())

    def web_search_tool(self, query):
        return f"[search results for '{query}' — placeholder summary]"
    def calc_tool(self, expr):
        try:
            return str(eval(expr))
        except Exception as e:
            return f"[calc error: {e}]"

    def maybe_summarize(self):
        cur = self.conn.cursor()
        cur.execute("SELECT COUNT(*) FROM memory")
        if cur.fetchone()[0] > self.summary_threshold:
            msgs = self.get_recent(limit=self.summary_threshold)
            convo = "\n".join([f"{r}: {c}" for r,c in msgs])
            summary = llm_call(f"Summarize this conversation in 2-3 sentences:\n{convo}")
            self.add_message("system", f"[summary] {summary}")
            cur.execute("DELETE FROM memory WHERE id NOT IN (SELECT id FROM memory ORDER BY id DESC LIMIT 5)")
            self.conn.commit()

    def handle_user(self, text):
        self.add_message("user", text)
        self.maybe_summarize()

        lt = text.lower()
        if lt.startswith("search:"):
            q = text.split(":",1)[1].strip()
            out = self.web_search_tool(q)
            self.add_message("assistant", out)
            return out
        if lt.startswith("calc:"):
            expr = text.split(":",1)[1].strip()
            out = self.calc_tool(expr)
            self.add_message("assistant", out)
            return out

        facts = self.get_facts()
        convo = "\n".join([f"{r}: {c}" for r,c in self.get_recent()])
        prompt = f"You are a concise assistant. Facts: {facts}\nConversation:\n{convo}\nUser: {text}\nAssistant:"
        reply = llm_call(prompt)
        self.add_message("assistant", reply)
        return reply

In [ ]:
agent = StatefulAgent()
print(agent.handle_user("Hello, I'm Hamsa!"))
agent.set_fact("name", "Hamsa")
print(agent.handle_user("search: hugging face transformers"))
print(agent.handle_user("calc: (5+3)**2"))
print(agent.handle_user("Do you remember my name?"))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

docs = [
    "Stateful agents maintain long-term memory using databases or embeddings.",
    "Context engineering involves prompt design, retrieval, and few-shot learning.",
    "Few-shot examples demonstrate correct response patterns to guide the model.",
    "Summarization helps reduce token usage and maintain relevant context."
]

vectorizer = TfidfVectorizer().fit(docs)
doc_vecs = vectorizer.transform(docs)

def retrieve(query, top_k=2):
    qv = vectorizer.transform([query])
    sims = linear_kernel(qv, doc_vecs).flatten()
    idxs = sims.argsort()[::-1][:top_k]
    return [docs[i] for i in idxs]

def smart_prompt(query):
    context = "\n".join(retrieve(query))
    return f"You are helpful. Use only this context:\n{context}\n\nQ: {query}\nA:"

print(agent.handle_user("Explain few-shot learning"))

In [ ]:
print("Facts:", agent.get_facts())
print("Recent messages:", agent.get_recent())
agent2 = StatefulAgent("agent_memory.sqlite")
print("Restored facts:", agent2.get_facts())